# D_8 training-size comparison

Train m_8 on nested 10k, 30k, and 80k subsets of D_8. Every model uses the same 20k held-out rows for validation and the same 100 initial sheet states for the policy comparison.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / 'data_generation.py').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Could not find repository root')
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import data_generation
from scratch import sequential_training as seq


In [2]:
N = 5
train_sizes = (10_000, 30_000, 80_000)
validation_size = 20_000
artifact_dir = repo_root / 'scratch' / 'sequential_experiment_100k'
output_dir = repo_root / 'scratch' / 'sequential_experiment_scaling'
output_dir.mkdir(parents=True, exist_ok=True)

# D_8.npz has 100k rows, enough for 80k training rows plus 20k validation rows.
d8 = seq.load_sequential_dataset(artifact_dir / 'D_8.npz')
splits = seq.fixed_validation_splits(
    d8, train_sizes=train_sizes, validation_size=validation_size, seed=8
)
m9 = seq.load_model(artifact_dir / 'm_9.npz')
states = data_generation.random_sheet_states(
    team1=4, team2=3, num_sims=100, rng=np.random.default_rng(8)
)
print(f'D_8 rows: {d8.final_scores.size}; validation rows: {validation_size}; comparison states: {states.x.shape[0]}')


D_8 rows: 100000; validation rows: 20000; comparison states: 100


In [3]:
models = {9: m9}
results = {}
for train_size, (train_data, validation_data) in splits.items():
    model, normalizer, training_info, validation = seq.train_model_with_validation(
        train_data, validation_data, seed=8, max_stones=2 * N,
        num_stones_per_side=N,
    )
    models[8] = (model, normalizer)
    validation_stats = seq.evaluate_model(model, normalizer, validation, N=N)
    model_searcher = seq.ThrowsGridSearcher(10, 10, 4)
    greedy_searcher = seq.GridAndRandomThrowSearcher(
        np.random.default_rng(0), grid_size=(10, 10, 4)
    )
    comparison = seq.compare_policies(
        8, states, models, max_stones=2 * N,
        searcher=greedy_searcher, model_searcher=model_searcher,
    )
    summary = seq.policy_summary(comparison)
    diagnostics = seq.model_candidate_diagnostics(
        states, model, normalizer, max_stones=2 * N,
        searcher=seq.ThrowsGridSearcher(10, 10, 4), team=1,
    )
    results[train_size] = (summary, validation_stats, training_info, diagnostics)
    seq.write_model(output_dir / f'm_8_{train_size}.npz', model, normalizer)
    seq.write_policy_comparison(output_dir / f'policy_comparison_8_{train_size}.npz', comparison)
    print(
        f'{train_size:>6,d} rows | '        f'model {summary["model_expected_score"]:.3f} | '        f'greedy {summary["greedy_expected_score"]:.3f} | '        f'diff {summary["expected_score_difference"]:.3f} '        f'± {summary["expected_score_difference_stderr"]:.3f} (stderr)'
    )
    print(
        f'       predicted team-0 diff: selected {diagnostics["selected_predicted_score"]:.3f}, '
        f'greedy {diagnostics["greedy_predicted_score"]:.3f}; '
        f'immediate realized team-0 diff: selected {diagnostics["selected_immediate_score"]:.3f}, '
        f'greedy {diagnostics["greedy_immediate_score"]:.3f}; '
        f'candidate corr {diagnostics["candidate_prediction_actual_correlation"]:.3f}'
    ),
    print(
        f'       terminal realized team-0 diff: selected {summary["model_expected_score"]:.3f}, '
        f'greedy {summary["greedy_expected_score"]:.3f}; '
        f'prediction error: selected '
        f'{summary["model_expected_score"] - diagnostics["selected_predicted_score"]:.3f}, '
        f'greedy {summary["greedy_expected_score"] - diagnostics["greedy_predicted_score"]:.3f}'
    )


10,000 rows | model -1.990 | greedy -2.000 | diff 0.010 ± 0.086 (stderr)
       predicted team-0 diff: selected -2.834, greedy -1.915; immediate realized team-0 diff: selected -1.030, greedy -2.550; candidate corr 0.391
       terminal realized team-0 diff: selected -1.990, greedy -2.000; prediction error: selected 0.844, greedy -0.085
30,000 rows | model -1.900 | greedy -2.000 | diff 0.100 ± 0.081 (stderr)
       predicted team-0 diff: selected -2.753, greedy -1.871; immediate realized team-0 diff: selected -1.130, greedy -2.550; candidate corr 0.347
       terminal realized team-0 diff: selected -1.900, greedy -2.000; prediction error: selected 0.853, greedy -0.129
80,000 rows | model -1.730 | greedy -2.000 | diff 0.270 ± 0.095 (stderr)
       predicted team-0 diff: selected -3.591, greedy -1.970; immediate realized team-0 diff: selected -0.490, greedy -2.550; candidate corr 0.386
       terminal realized team-0 diff: selected -1.730, greedy -2.000; prediction error: selected 1.861, 

In [4]:
# The validation statistics use the identical 20k rows for all three models.
for train_size, (_, validation_stats, _, _) in results.items():
    print(f'\n{train_size:,} training rows')
    print(validation_stats)



10,000 training rows
NeuralNetStats(r_squared=Estimate(value=0.19653335038335717, stderr=0.00020271290299903014), correct_score_probability=Estimate(value=0.5245962747583122, stderr=0.0018437627332652435), calibration=(CalibrationBucket(lower_bound=0.0, upper_bound=0.1, count=176273, predicted_fraction=0.005141425674005168, predicted_stderr=3.5561397806851446e-05, actual_fraction=0.006359453801773386, actual_stderr=0.00018933586790168357), CalibrationBucket(lower_bound=0.1, upper_bound=0.2, count=8253, predicted_fraction=0.14855632152008538, predicted_stderr=0.0003172370838922129, actual_fraction=0.16733309099721314, actual_stderr=0.004109102360283435), CalibrationBucket(lower_bound=0.2, upper_bound=0.3, count=6880, predicted_fraction=0.2475927018373998, predicted_stderr=0.00034502212535280973, actual_fraction=0.2742732558139535, actual_stderr=0.005379174588687812), CalibrationBucket(lower_bound=0.3, upper_bound=0.4, count=5936, predicted_fraction=0.3500865369470978, predicted_stderr=